In [0]:
dbutils.widgets.text("environment", "dev")
dbutils.widgets.text("pipeline_run_id", "")
dbutils.widgets.text("catalog_name", "retail_lakehouse")

In [0]:
import uuid
from datetime import datetime

environment = dbutils.widgets.get("environment").strip()
pipeline_run_id = dbutils.widgets.get("pipeline_run_id").strip()
catalog_name = dbutils.widgets.get("catalog_name").strip()

if not pipeline_run_id:
    pipeline_run_id = str(uuid.uuid4())

start_timestamp = datetime.now()

print(f"Environment     : {environment}")
print(f"Pipeline run ID : {pipeline_run_id}")
print(f"Catalog         : {catalog_name}")

In [0]:

sales_enriched_table = (
    f"{catalog_name}.silver.sales_enriched"
)

silver_product_table = (
    f"{catalog_name}.silver.products"
)

silver_customer_table = (
    f"{catalog_name}.silver.customers"
)

silver_store_table = (
    f"{catalog_name}.silver.stores"
)

silver_promotion_table = (
    f"{catalog_name}.silver.promotions"
)

silver_inventory_table = (
    f"{catalog_name}.silver.inventory"
)

gold_fact_sales_table = (
    f"{catalog_name}.gold.fact_sales"
)

gold_dim_product_table = (
    f"{catalog_name}.gold.dim_product"
)

gold_dim_customer_table = (
    f"{catalog_name}.gold.dim_customer"
)

gold_dim_store_table = (
    f"{catalog_name}.gold.dim_store"
)

gold_dim_promotion_table = (
    f"{catalog_name}.gold.dim_promotion"
)

gold_dim_date_table = (
    f"{catalog_name}.gold.dim_date"
)

gold_fact_inventory_table = (
    f"{catalog_name}.gold.fact_inventory"
)

audit_table = (
    f"{catalog_name}.audit.pipeline_runs"
)

In [0]:
required_tables = [
    sales_enriched_table,
    silver_product_table,
    silver_customer_table,
    silver_store_table,
    silver_promotion_table,
    silver_inventory_table
]

missing_tables = [
    table_name
    for table_name in required_tables
    if not spark.catalog.tableExists(table_name)
]

if missing_tables:
    raise RuntimeError(
        f"Required Silver tables are missing: {missing_tables}"
    )

print("All required Silver tables are available.")

In [0]:
#Building dimension Product:

from pyspark.sql import functions as F
from pyspark.sql.window import Window

products_df = spark.table(silver_product_table)



In [0]:
dim_product_df = (
    products_df
    .select(
        "product_id",
        "product_name",
        "category",
        "brand",
        "standard_price",
        "cost_price",
        "product_status"
    )
    .dropDuplicates(["product_id"])
    .withColumn(
        "product_key",
        F.xxhash64("product_id")
    )
    .withColumn(
        "_gold_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_gold_updated_timestamp",
        F.current_timestamp()
    )
)

In [0]:
dim_product_df = dim_product_df.select(
    "product_key",
    "product_id",
    "product_name",
    "category",
    "brand",
    "standard_price",
    "cost_price",
    "product_status",
    "_gold_pipeline_run_id",
    "_gold_updated_timestamp"
)

In [0]:
(
    dim_product_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_dim_product_table)
)

print(
    f"Created {gold_dim_product_table}: "
    f"{dim_product_df.count()} rows"
)

In [0]:
#Building Dimension Customer:

customers_df = spark.table(silver_customer_table)

dim_customer_df = (
    customers_df
    .select(
        "customer_id",
        "customer_name",
        "age",
        "customer_age_group",
        "city",
        "state",
        "region",
        "loyalty_tier",
        "registration_date"
    )
    .dropDuplicates(["customer_id"])
    .withColumn(
        "customer_key",
        F.xxhash64("customer_id")
    )
    .withColumn(
        "_gold_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_gold_updated_timestamp",
        F.current_timestamp()
    )
)

dim_customer_df = dim_customer_df.select(
    "customer_key",
    "customer_id",
    "customer_name",
    "age",
    "customer_age_group",
    "city",
    "state",
    "region",
    "loyalty_tier",
    "registration_date",
    "_gold_pipeline_run_id",
    "_gold_updated_timestamp"
)

(
    dim_customer_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_dim_customer_table)
)

In [0]:
sales_enriched_df = spark.table(sales_enriched_table)

null_customer_count = (
    sales_enriched_df
    .filter(F.col("customer_id").isNull())
    .count()
)

print(f"Null customer transactions: {null_customer_count}")

In [0]:
customer_schema = spark.table(
    gold_dim_customer_table
).schema

unknown_customer_data = [{
    "customer_key": -1,
    "customer_id": "UNKNOWN",
    "customer_name": "Unknown Customer",
    "age": None,
    "customer_age_group": "Unknown",
    "city": "Unknown",
    "state": "Unknown",
    "region": "Unknown",
    "loyalty_tier": "Unknown",
    "registration_date": None,
    "_gold_pipeline_run_id": pipeline_run_id,
    "_gold_updated_timestamp": datetime.now()
}]

unknown_customer_df = spark.createDataFrame(
    unknown_customer_data,
    schema=customer_schema
)

In [0]:
unknown_exists = (
    spark.table(gold_dim_customer_table)
    .filter(F.col("customer_key") == -1)
    .limit(1)
    .count()
    > 0
)

if not unknown_exists:
    (
        unknown_customer_df
        .write
        .format("delta")
        .mode("append")
        .saveAsTable(gold_dim_customer_table)
    )

In [0]:
#Building Dimension Store:

stores_df = spark.table(silver_store_table)

dim_store_df = (
    stores_df
    .select(
        "store_id",
        "store_name",
        "city",
        "state",
        "region",
        "store_size"
    )
    .dropDuplicates(["store_id"])
    .withColumn(
        "store_key",
        F.xxhash64("store_id")
    )
    .withColumn(
        "_gold_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_gold_updated_timestamp",
        F.current_timestamp()
    )
    .select(
        "store_key",
        "store_id",
        "store_name",
        "city",
        "state",
        "region",
        "store_size",
        "_gold_pipeline_run_id",
        "_gold_updated_timestamp"
    )
)

(
    dim_store_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_dim_store_table)
)

In [0]:
#Building Dimension Promotion

promotions_df = spark.table(silver_promotion_table)

dim_promotion_df = (
    promotions_df
    .select(
        "promotion_id",
        "promotion_name",
        "category",
        "discount_type",
        "discount_value",
        "start_date",
        "end_date",
        "promotion_status"
    )
    .dropDuplicates(["promotion_id"])
    .withColumn(
        "promotion_key",
        F.xxhash64("promotion_id")
    )
    .withColumn(
        "_gold_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_gold_updated_timestamp",
        F.current_timestamp()
    )
    .select(
        "promotion_key",
        "promotion_id",
        "promotion_name",
        "category",
        "discount_type",
        "discount_value",
        "start_date",
        "end_date",
        "promotion_status",
        "_gold_pipeline_run_id",
        "_gold_updated_timestamp"
    )
)

(
    dim_promotion_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_dim_promotion_table)
)

In [0]:
#Adding an unknown promotrion row:

promotion_schema = spark.table(
    gold_dim_promotion_table
).schema

unknown_promotion_data = [{
    "promotion_key": -1,
    "promotion_id": "NO_PROMOTION",
    "promotion_name": "No Promotion",
    "category": "Not Applicable",
    "discount_type": "NONE",
    "discount_value": None,
    "start_date": None,
    "end_date": None,
    "promotion_status": "NOT_APPLICABLE",
    "_gold_pipeline_run_id": pipeline_run_id,
    "_gold_updated_timestamp": datetime.now()
}]

unknown_promotion_df = spark.createDataFrame(
    unknown_promotion_data,
    schema=promotion_schema
)

(
    unknown_promotion_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable(gold_dim_promotion_table)
)

In [0]:
#Creating Date Dimension:

sales_date_range = (
    sales_enriched_df
    .agg(
        F.min("transaction_date").alias("minimum_date"),
        F.max("transaction_date").alias("maximum_date")
    )
    .first()
)

minimum_date = sales_date_range["minimum_date"]
maximum_date = sales_date_range["maximum_date"]

if minimum_date is None or maximum_date is None:
    raise RuntimeError(
        "Cannot create date dimension because sales dates are unavailable."
    )

print(f"Minimum date: {minimum_date}")
print(f"Maximum date: {maximum_date}")

In [0]:
date_start = F.date_sub(
    F.lit(minimum_date),
    30
)

date_end = F.add_months(
    F.lit(maximum_date),
    12
)

In [0]:
dim_date_df = (
    spark.range(1)
    .select(
        F.explode(
            F.sequence(
                date_start,
                date_end,
                F.expr("INTERVAL 1 DAY")
            )
        ).alias("full_date")
    )
)

In [0]:
dim_date_df = (
    dim_date_df
    .withColumn(
        "date_key",
        F.date_format("full_date", "yyyyMMdd").cast("integer")
    )
    .withColumn("calendar_year", F.year("full_date"))
    .withColumn("calendar_quarter", F.quarter("full_date"))
    .withColumn("calendar_month", F.month("full_date"))
    .withColumn(
        "month_name",
        F.date_format("full_date", "MMMM")
    )
    .withColumn(
        "month_short_name",
        F.date_format("full_date", "MMM")
    )
    .withColumn(
        "year_month",
        F.date_format("full_date", "yyyy-MM")
    )
    .withColumn(
        "week_of_year",
        F.weekofyear("full_date")
    )
    .withColumn(
        "day_of_month",
        F.dayofmonth("full_date")
    )
    .withColumn(
        "day_of_week_number",
        F.dayofweek("full_date")
    )
    .withColumn(
        "day_name",
        F.date_format("full_date", "EEEE")
    )
    .withColumn(
        "is_weekend",
        F.when(
            F.dayofweek("full_date").isin(1, 7),
            1
        ).otherwise(0)
    )
    .withColumn(
        "financial_year",
        F.when(
            F.month("full_date") >= 4,
            F.concat(
                F.year("full_date"),
                F.lit("-"),
                F.year("full_date") + 1
            )
        ).otherwise(
            F.concat(
                F.year("full_date") - 1,
                F.lit("-"),
                F.year("full_date")
            )
        )
    )
    .withColumn(
        "_gold_pipeline_run_id",
        F.lit(pipeline_run_id)
    )
    .withColumn(
        "_gold_updated_timestamp",
        F.current_timestamp()
    )
)

In [0]:
(
    dim_date_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_dim_date_table)
)

In [0]:
#Creating Fact Sales:
#Reading Gold dimensions for key lookup:

gold_products_df = (
    spark.table(gold_dim_product_table)
    .select("product_id", "product_key")
)

gold_customers_df = (
    spark.table(gold_dim_customer_table)
    .select("customer_id", "customer_key")
)

gold_stores_df = (
    spark.table(gold_dim_store_table)
    .select("store_id", "store_key")
)

gold_promotions_df = (
    spark.table(gold_dim_promotion_table)
    .select("promotion_id", "promotion_key")
)

In [0]:
#Joining surrogate keys into sales:

fact_sales_df = (
    sales_enriched_df.alias("sales")
    .join(
        gold_products_df.alias("product"),
        on="product_id",
        how="left"
    )
    .join(
        gold_customers_df.alias("customer"),
        on="customer_id",
        how="left"
    )
    .join(
        gold_stores_df.alias("store"),
        on="store_id",
        how="left"
    )
    .join(
        gold_promotions_df.alias("promotion"),
        on="promotion_id",
        how="left"
    )
)

In [0]:
#Assigning Unknwon keys:

fact_sales_df = (
    fact_sales_df
    .withColumn(
        "product_key",
        F.coalesce(
            F.col("product_key"),
            F.lit(-1)
        )
    )
    .withColumn(
        "customer_key",
        F.coalesce(
            F.col("customer_key"),
            F.lit(-1)
        )
    )
    .withColumn(
        "store_key",
        F.coalesce(
            F.col("store_key"),
            F.lit(-1)
        )
    )
    .withColumn(
        "promotion_key",
        F.coalesce(
            F.col("promotion_key"),
            F.lit(-1)
        )
    )
    .withColumn(
        "date_key",
        F.date_format(
            "transaction_date",
            "yyyyMMdd"
        ).cast("integer")
    )
)

In [0]:
#Selecting the fact table grain, 
#Grain is: One row per unique retail transaction.
fact_sales_df = (
    fact_sales_df
    .select(
        "transaction_id",
        "date_key",
        "product_key",
        "customer_key",
        "store_key",
        "promotion_key",
        "transaction_timestamp",
        "quantity",
        "unit_price",
        "discount_amount",
        "gross_amount",
        "net_amount",
        "recognized_sales_amount",
        "estimated_cost_amount",
        "gross_profit_amount",
        "transaction_status",
        "payment_method",
        "sales_channel",
        "return_flag",
        "cancellation_flag",
        "_source_bronze_run_id",
        "_silver_pipeline_run_id",
        F.lit(pipeline_run_id)
            .alias("_gold_pipeline_run_id"),
        F.current_timestamp()
            .alias("_gold_updated_timestamp")
    )
)

In [0]:
#Validating Uniqueness:

duplicate_fact_count = (
    fact_sales_df
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

if duplicate_fact_count > 0:
    raise RuntimeError(
        f"fact_sales contains {duplicate_fact_count} "
        "duplicate transaction IDs."
    )

print("fact_sales grain validation passed.")

In [0]:
(
    fact_sales_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_fact_sales_table)
)

In [0]:
#Building the Inventory Fact Table:

inventory_df = spark.table(silver_inventory_table)

fact_inventory_df = (
    inventory_df.alias("inventory")
    .join(
        gold_products_df.alias("product"),
        on="product_id",
        how="left"
    )
    .join(
        gold_stores_df.alias("store"),
        on="store_id",
        how="left"
    )
    .withColumn(
        "date_key",
        F.date_format(
            "inventory_date",
            "yyyyMMdd"
        ).cast("integer")
    )
    .withColumn(
        "product_key",
        F.coalesce(
            F.col("product_key"),
            F.lit(-1)
        )
    )
    .withColumn(
        "store_key",
        F.coalesce(
            F.col("store_key"),
            F.lit(-1)
        )
    )
    .withColumn(
        "stock_out_flag",
        F.when(
            F.col("closing_stock") == 0,
            1
        ).otherwise(0)
    )
    .withColumn(
        "low_stock_flag",
        F.when(
            (F.col("closing_stock") > 0)
            & (F.col("closing_stock") <= 20),
            1
        ).otherwise(0)
    )
)

fact_inventory_df = fact_inventory_df.select(
    "date_key",
    "product_key",
    "store_key",
    "inventory_date",
    "opening_stock",
    "received_quantity",
    "sold_quantity",
    "damaged_quantity",
    "closing_stock",
    "stock_out_flag",
    "low_stock_flag",
    F.lit(pipeline_run_id)
        .alias("_gold_pipeline_run_id"),
    F.current_timestamp()
        .alias("_gold_updated_timestamp")
)

(
    fact_inventory_df
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(gold_fact_inventory_table)
)

In [0]:
#Validating the Gold Model:

gold_tables = [
    gold_dim_product_table,
    gold_dim_customer_table,
    gold_dim_store_table,
    gold_dim_promotion_table,
    gold_dim_date_table,
    gold_fact_sales_table,
    gold_fact_inventory_table
]

for table_name in gold_tables:
    print(
        f"{table_name}: "
        f"{spark.table(table_name).count()} rows"
    )

In [0]:
#Checking Orphan Dimension Keys:

fact_sales_validation_df = spark.table(
    gold_fact_sales_table
)

fact_sales_validation_df.select(
    F.sum(
        F.when(F.col("product_key") == -1, 1)
        .otherwise(0)
    ).alias("unknown_products"),
    F.sum(
        F.when(F.col("customer_key") == -1, 1)
        .otherwise(0)
    ).alias("unknown_customers"),
    F.sum(
        F.when(F.col("store_key") == -1, 1)
        .otherwise(0)
    ).alias("unknown_stores"),
    F.sum(
        F.when(F.col("promotion_key") == -1, 1)
        .otherwise(0)
    ).alias("no_or_unknown_promotions")
).show()

In [0]:
display(
    spark.table("retail_lakehouse.gold.fact_sales")
    .groupBy("date_key")
    .count()
    .orderBy("date_key")
)